# Image Similarity Search with KNN

Find visually similar real images.

## Step 1: Import libraries

NearestNeighbors performs similarity search.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

from sklearn.neighbors import NearestNeighbors

## Step 2: Load the gallery

The gallery contains five real-image categories.

In [ ]:
DATASET_DIR=Path("../datasets/07_image_similarity_gallery")
def load_images(path, size=(128,128)):
    images, labels = [], []
    for class_dir in sorted(path.iterdir()):
        if class_dir.is_dir():
            for fp in sorted(class_dir.glob("*.png")):
                images.append(np.array(Image.open(fp).convert("RGB").resize(size)))
                labels.append(class_dir.name)
    return np.array(images), np.array(labels)

images,labels=load_images(DATASET_DIR)
print(images.shape)

## Step 3: Extract colour features

Similar colour distributions create nearby vectors.

In [ ]:
def color_hist(im,bins=16):
    f=[]
    for ch in range(3):
        h,_=np.histogram(im[:,:,ch],bins=bins,range=(0,256),density=True); f.extend(h)
    return np.array(f)
features=np.array([color_hist(im) for im in images])
print(features.shape)


## Step 4: Build the search index

NearestNeighbors stores gallery features.

In [ ]:
search=NearestNeighbors(n_neighbors=6).fit(features)

## Step 5: Query and display matches

The first match is the query itself.

In [ ]:
q=3
dist,idx=search.kneighbors([features[q]])
plt.figure(figsize=(15,3))
for p,j in enumerate(idx[0],1):
 plt.subplot(1,6,p); plt.imshow(images[j]); plt.title(f"{labels[j]}\n{dist[0][p-1]:.3f}"); plt.axis("off")
plt.show()

# Search with One Separate Query Image

This notebook performs image retrieval instead of normal classification.

The single query image is converted into the same colour-histogram feature vector used to build the nearest-neighbour search index. The closest gallery images are then displayed.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

query_image_path = Path(
    "../datasets/07_image_similarity_gallery/coffee/000_original.png"
)

query_image = Image.open(query_image_path).convert("RGB")
query_image = query_image.resize((128, 128))
query_image_array = np.array(query_image)

query_features = extract_color_histogram(query_image_array)
query_features_2d = query_features.reshape(1, -1)

number_of_matches = 5

distances, indices = search_model.kneighbors(
    query_features_2d,
    n_neighbors=number_of_matches,
)

plt.figure(figsize=(15, 3))

plt.subplot(1, number_of_matches + 1, 1)
plt.imshow(query_image_array)
plt.title("Query Image")
plt.axis("off")

for position, image_index in enumerate(indices[0], start=2):
    plt.subplot(1, number_of_matches + 1, position)
    plt.imshow(images[image_index])
    plt.title(
        f"{labels[image_index]}\n"
        f"Distance={distances[0][position - 2]:.4f}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()
